# 中证800 V54 Governed Dynamic Factor Pool LGB 实验

目标：验证“带约束、带衰减、带因子池治理”的动态因子选择，是否能比固定 JQ 因子池 LGB 更鲁棒。

核心约束：
- 固定 `alpha_1m` label 和 LGB 回归，不引入新 label，不做 early stopping。
- 不做自由动态选因子，先做长期稳定性、近期适配度、方向稳定、最差年份、覆盖率、相关性和风格拥挤治理。
- 只保留三条模型线：`fixed_jq_baseline_lgb`、`governed_dynamic_pool_lgb`、`core_plus_adaptive_lgb`。
- 年度滚动验证：`2016-2022 -> 2023`、`2017-2023 -> 2024`、`2018-2024 -> 2025`、`2019-2025 -> 2026`。
- 输出月度收益、年度汇总、RankIC、因子入选原因、剔除原因、group 暴露和相对 baseline 差值。

注意：本 notebook 的数据重建模块需要在聚宽研究环境运行；如果 `REBUILD_DATA=False` 且本地已有 `DATA_PATH`，后续分析可以直接复用缓存。

In [ ]:
import os
import gc
import pickle
import datetime
import numpy as np
import pandas as pd

try:
    import lightgbm as lgb
except Exception as err:
    lgb = None
    print("lightgbm import failed:", err)

try:
    from jqdata import *
    from jqfactor import get_factor_values
except Exception as err:
    print("JoinQuant imports failed. Cached DATA_PATH can still be used if it exists. err=", err)

UNIVERSE_INDEX = "000906.XSHG"
BENCHMARK_INDEX = "000906.XSHG"
OUT_DIR = "csi800_ml_v54_governed_dynamic_factor_pool_lgb_outputs"
CACHE_DIR = os.path.join(OUT_DIR, "monthly_factor_cache_jq_only_v1")
DATA_PATH = os.path.join(OUT_DIR, "csi800_jq_factor_monthly_panel.csv")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

REBUILD_DATA = True
FORCE_REFETCH_MONTHLY_CACHE = False
PRECHECK_FACTORS = True
DATA_START_DATE = "2016-01-01"
LABEL_END_DATE = "2026-05-31"
MIN_LISTING_DAYS = 180
FACTOR_CHUNK_SIZE = 10
PRECHECK_STOCK_SAMPLE = 40

# User examples are 2016-2022 -> 2023, 2017-2023 -> 2024, etc.
# This is 7 calendar years inclusive. Change to 6 only if you want 2017-2022 -> 2023.
TRAIN_START_YEAR_OFFSET = 7
EVAL_YEARS = [2023, 2024, 2025, 2026]

TARGET_COL = "alpha_1m"
RAW_RETURN_COL = "raw_return_1m"
BENCHMARK_RETURN_COL = "benchmark_csi800_1m"
TOP_LIST = [10, 20]

MIN_FACTOR_COVERAGE = 0.70
MIN_MONTHS_FOR_FACTOR = 36
MAX_ABS_CORR = 0.82
TARGET_FACTOR_COUNT = 24
MIN_FACTOR_COUNT = 18
CORE_TARGET_COUNT = 16
ADAPTIVE_TARGET_COUNT = 8
MAX_DEFENSIVE_LOW_VOL_LIQ_COUNT = 7

# Governance weights: long-term quality dominates; recent performance is only a small regime adapter.
LONG_SCORE_WEIGHT = 0.60
MEDIUM_SCORE_WEIGHT = 0.25
RECENT_SCORE_WEIGHT = 0.15

EXPORT_MODELS = False
MODEL_EXPORT_DIR = os.path.join(OUT_DIR, "exported_models")
os.makedirs(MODEL_EXPORT_DIR, exist_ok=True)

LGB_PARAMS = {
    "objective": "regression",
    "metric": "l2",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 200,
    "feature_fraction": 1.0,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 0.3,
    "verbose": -1,
}
NUM_BOOST_ROUND = 120

print("OUT_DIR:", OUT_DIR)
print("DATA_PATH:", DATA_PATH)


In [ ]:
def unique_keep_order(cols):
    seen = set()
    out = []
    for c in cols:
        if c not in seen:
            out.append(c)
            seen.add(c)
    return out


KNOWN_BAD_FACTORS = set([
    "market_beta", "resvol", "relative_momentum", "profit",
    "long_growth", "earnvar", "financial_leverage",
])

FACTOR_GROUPS = {
    "style": [
        "size", "non_linear_size", "beta", "residual_volatility",
        "liquidity", "earnings_yield", "growth", "leverage", "momentum",
    ],
    "value_cashflow": [
        "cash_flow_to_price_ratio", "book_to_price_ratio", "sales_to_price_ratio",
        "cash_earnings_to_price_ratio", "earnings_to_price_ratio", "cfo_to_ev",
    ],
    "quality_profit": [
        "roe_ttm", "roa_ttm", "ROAEBITTTM", "net_profit_ratio",
        "operating_profit_ratio", "profit_margin_ttm",
        "net_profit_to_total_operate_revenue_ttm", "operating_profit_to_total_profit",
        "adjusted_profit_to_total_profit", "net_operating_cash_flow_coverage",
        "cash_rate_of_sales", "goods_service_cash_to_operating_revenue_ttm",
        "net_operate_cash_flow_to_asset", "gross_profit_ttm",
        "operating_profit_per_share", "net_operate_cash_flow_per_share",
        "total_operating_revenue_per_share",
    ],
    "growth_balance": [
        "ACCA", "growth", "operating_revenue_growth_rate", "total_profit_growth_rate",
        "np_parent_company_owners_growth_rate", "net_profit_growth_rate",
        "net_operate_cashflow_growth_rate", "total_asset_growth_rate", "net_asset_growth_rate",
        "MLEV", "debt_to_equity_ratio", "debt_to_asset_ratio",
        "debt_to_tangible_equity_ratio", "super_quick_ratio", "net_working_capital",
    ],
    "momentum_risk": [
        "momentum", "Rank1M", "sharpe_ratio_60", "beta", "residual_volatility",
        "Variance20", "Variance60", "Variance120",
    ],
    "volume_technical": [
        "VOL5", "VOL10", "VOL20", "VOL60", "VOL120", "DAVOL5", "DAVOL10", "DAVOL20",
        "turnover_volatility", "VMACD", "VOSC", "MFI14", "ATR6", "ATR14", "MACDC",
    ],
    "shape_distribution": [
        "Skewness20", "Skewness60", "Skewness120", "Kurtosis20", "Kurtosis60", "Kurtosis120",
    ],
}

GROUP_MAX_QUOTA = {
    "style": 2,
    "value_cashflow": 5,
    "quality_profit": 5,
    "growth_balance": 2,
    "momentum_risk": 4,
    "volume_technical": 4,
    "shape_distribution": 2,
}

GROUP_MIN_QUOTA = {
    "value_cashflow": 3,
    "quality_profit": 2,
    "momentum_risk": 2,
    "volume_technical": 2,
}

FIXED_JQ_BASELINE_FACTORS = [
    "cash_flow_to_price_ratio", "book_to_price_ratio", "earnings_yield", "sales_to_price_ratio",
    "cash_earnings_to_price_ratio", "earnings_to_price_ratio", "roe_ttm", "roa_ttm",
    "gross_profit_ttm", "operating_profit_to_total_profit",
    "net_operate_cash_flow_to_total_liability", "net_operating_cash_flow_coverage",
    "adjusted_profit_to_total_profit", "ACCA", "growth", "net_working_capital",
    "operating_profit_per_share", "net_operate_cash_flow_per_share",
    "total_operating_revenue_per_share", "super_quick_ratio", "MLEV", "debt_to_equity_ratio",
    "debt_to_tangible_equity_ratio", "momentum", "Rank1M", "sharpe_ratio_60", "Variance20",
    "liquidity", "beta", "MFI14", "DAVOL10", "VOL10", "VMACD", "VOSC",
    "Skewness20", "Kurtosis20", "Kurtosis60",
]

LOW_VOL_LIQ_FACTORS = set([
    "liquidity", "residual_volatility", "beta", "Variance20", "Variance60", "Variance120",
    "VOL5", "VOL10", "VOL20", "VOL60", "VOL120", "DAVOL5", "DAVOL10", "DAVOL20",
    "turnover_volatility", "ATR6", "ATR14",
])

_candidate_factor_parts = []
for _group_name in FACTOR_GROUPS:
    _candidate_factor_parts.extend(FACTOR_GROUPS[_group_name])
_candidate_factor_parts.extend(FIXED_JQ_BASELINE_FACTORS)
ALL_CANDIDATE_FACTORS = unique_keep_order(_candidate_factor_parts)
ALL_CANDIDATE_FACTORS = [f for f in ALL_CANDIDATE_FACTORS if f not in KNOWN_BAD_FACTORS]

FACTOR_TO_GROUP = {}
for group_name, cols in FACTOR_GROUPS.items():
    for col in cols:
        if col not in FACTOR_TO_GROUP:
            FACTOR_TO_GROUP[col] = group_name


def risk_bucket(factor):
    if factor in LOW_VOL_LIQ_FACTORS:
        return "defensive_low_vol_liq"
    return "other"

print("candidate factor count:", len(ALL_CANDIDATE_FACTORS))
print(ALL_CANDIDATE_FACTORS)


In [ ]:
def chunks(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def first_trade_day_by_month(trade_days):
    out = []
    last_month = None
    for dt in sorted(pd.to_datetime(trade_days)):
        month = dt.strftime("%Y-%m")
        if month != last_month:
            out.append(dt)
            last_month = month
    return out


def build_month_schedule(start_date, label_end_date):
    start_ts = pd.Timestamp(start_date)
    buffer_start = (start_ts - pd.Timedelta(days=60)).strftime("%Y-%m-%d")
    all_days = list(pd.to_datetime(get_trade_days(start_date=buffer_start, end_date=label_end_date)))
    if len(all_days) == 0:
        raise ValueError("no trade days")
    first_days = first_trade_day_by_month([d for d in all_days if d >= start_ts])
    rows = []
    day_to_pos = {pd.Timestamp(d): i for i, d in enumerate(all_days)}
    for i in range(len(first_days) - 1):
        rebalance = pd.Timestamp(first_days[i])
        next_date = pd.Timestamp(first_days[i + 1])
        pos = day_to_pos.get(rebalance, None)
        if pos is None or pos <= 0:
            continue
        feature_date = pd.Timestamp(all_days[pos - 1])
        if next_date > pd.Timestamp(label_end_date):
            continue
        rows.append({
            "rebalance_date": rebalance,
            "feature_date": feature_date,
            "next_date": next_date,
        })
    return pd.DataFrame(rows)


def filter_listed_days(stock_list, feature_date, min_days):
    out = []
    for stock in stock_list:
        try:
            info = get_security_info(stock)
            if feature_date.date() - info.start_date >= datetime.timedelta(days=min_days):
                out.append(stock)
        except Exception:
            pass
    return out


def filter_st_on_date(stock_list, date):
    if len(stock_list) == 0:
        return []
    try:
        st_df = get_extras("is_st", stock_list, start_date=date, end_date=date, df=True)
        if st_df is None or st_df.empty:
            return stock_list
        s = st_df.iloc[0, :]
        return [stock for stock in stock_list if stock in s.index and not bool(s[stock])]
    except Exception as err:
        print("ST filter skipped on {} err={}".format(date, err))
        return stock_list


def precheck_valid_factors(schedule, factor_cols):
    factor_cols = unique_keep_order([f for f in factor_cols if f not in KNOWN_BAD_FACTORS])
    manifest_path = os.path.join(OUT_DIR, "v54_factor_validity_manifest.csv")
    if not PRECHECK_FACTORS:
        return factor_cols
    if os.path.exists(manifest_path) and not FORCE_REFETCH_MONTHLY_CACHE:
        old = pd.read_csv(manifest_path)
        if "factor" in old.columns and "valid" in old.columns:
            old_factor_set = set(list(old["factor"]))
            if set(factor_cols).issubset(old_factor_set):
                valid = list(old[old["valid"] == True]["factor"])
                print("loaded factor validity manifest", manifest_path, "valid", len(valid))
                return [f for f in factor_cols if f in set(valid)]
            print("factor validity manifest is stale; rebuilding", manifest_path)

    if schedule.empty:
        raise ValueError("empty schedule for factor precheck")
    feature_date = pd.Timestamp(schedule.iloc[0]["feature_date"])
    stocks = get_index_stocks(UNIVERSE_INDEX, feature_date.strftime("%Y-%m-%d"))
    stocks = filter_listed_days(stocks, feature_date, MIN_LISTING_DAYS)[:PRECHECK_STOCK_SAMPLE]
    if len(stocks) == 0:
        raise ValueError("no stocks for factor precheck")
    date_str = feature_date.strftime("%Y-%m-%d")
    rows = []
    valid = []
    for factor in factor_cols:
        ok = False
        err_msg = ""
        try:
            data = get_factor_values(stocks, [factor], end_date=date_str, count=1)
            ok = data is not None and factor in data
            if ok:
                valid.append(factor)
        except Exception as err:
            err_msg = str(err)
        rows.append({"factor": factor, "valid": bool(ok), "error": err_msg})
    pd.DataFrame(rows).to_csv(manifest_path, index=False)
    print("factor precheck valid/total:", len(valid), len(factor_cols))
    print("invalid factors:", [r["factor"] for r in rows if not r["valid"]])
    return valid


def fetch_factor_snapshot(stock_list, factor_cols, feature_date):
    out = pd.DataFrame(index=stock_list)
    if len(stock_list) == 0 or len(factor_cols) == 0:
        return out
    date_str = pd.Timestamp(feature_date).strftime("%Y-%m-%d")
    for factor_chunk in chunks(factor_cols, FACTOR_CHUNK_SIZE):
        try:
            factor_data = get_factor_values(stock_list, factor_chunk, end_date=date_str, count=1)
        except Exception as err:
            print("factor chunk failed", date_str, factor_chunk, err)
            factor_data = None
        for factor in factor_chunk:
            try:
                if factor_data is not None and factor in factor_data:
                    out[factor] = factor_data[factor].iloc[0, :].reindex(stock_list)
                else:
                    one = get_factor_values(stock_list, [factor], end_date=date_str, count=1)
                    if one is None or factor not in one:
                        out[factor] = np.nan
                    else:
                        out[factor] = one[factor].iloc[0, :].reindex(stock_list)
            except Exception as err:
                print("factor failed", date_str, factor, err)
                out[factor] = np.nan
        gc.collect()
    return out.reindex(index=stock_list, columns=factor_cols)


def fetch_forward_close_return(stock_list, start_date, end_date):
    out = pd.Series(index=stock_list, dtype=float)
    if len(stock_list) == 0:
        return out
    try:
        px = get_price(
            stock_list,
            start_date=pd.Timestamp(start_date).strftime("%Y-%m-%d"),
            end_date=pd.Timestamp(end_date).strftime("%Y-%m-%d"),
            frequency="daily",
            fields=["close"],
            skip_paused=False,
            fq="pre",
            panel=False,
            fill_paused=True,
        )
    except Exception as err:
        print("stock return fetch failed", start_date, end_date, err)
        return out
    if px is None or px.empty:
        return out
    px["time"] = pd.to_datetime(px["time"]).dt.normalize()
    mat = px.pivot_table(index="time", columns="code", values="close").sort_index()
    if mat.empty or len(mat) < 2:
        return out
    ret = mat.iloc[-1] / mat.iloc[0] - 1
    return ret.reindex(stock_list)


def fetch_index_close_return(index_code, start_date, end_date):
    try:
        px = get_price(
            index_code,
            start_date=pd.Timestamp(start_date).strftime("%Y-%m-%d"),
            end_date=pd.Timestamp(end_date).strftime("%Y-%m-%d"),
            frequency="daily",
            fields=["close"],
            fq="pre",
            panel=False,
        )
    except Exception as err:
        print("index return fetch failed", start_date, end_date, err)
        return np.nan
    if px is None or len(px) < 2:
        return np.nan
    close = pd.Series(px["close"]).astype(float)
    return float(close.iloc[-1] / close.iloc[0] - 1)


def build_one_month_panel(row, valid_factor_cols):
    rebalance_date = pd.Timestamp(row["rebalance_date"])
    feature_date = pd.Timestamp(row["feature_date"])
    next_date = pd.Timestamp(row["next_date"])
    cache_path = os.path.join(CACHE_DIR, "panel_{}.csv".format(rebalance_date.strftime("%Y%m%d")))
    if os.path.exists(cache_path) and not FORCE_REFETCH_MONTHLY_CACHE:
        return pd.read_csv(cache_path)

    stock_list = get_index_stocks(UNIVERSE_INDEX, feature_date.strftime("%Y-%m-%d"))
    stock_list = filter_listed_days(stock_list, feature_date, MIN_LISTING_DAYS)
    stock_list = filter_st_on_date(stock_list, feature_date.strftime("%Y-%m-%d"))
    if len(stock_list) == 0:
        return pd.DataFrame()

    factor_df = fetch_factor_snapshot(stock_list, valid_factor_cols, feature_date)
    factor_df.insert(0, "stock", factor_df.index)
    factor_df["rebalance_date"] = rebalance_date
    factor_df["feature_date"] = feature_date
    factor_df["next_date"] = next_date

    stock_ret = fetch_forward_close_return(stock_list, rebalance_date, next_date)
    bench_ret = fetch_index_close_return(BENCHMARK_INDEX, rebalance_date, next_date)
    factor_df[RAW_RETURN_COL] = factor_df["stock"].map(stock_ret)
    factor_df[BENCHMARK_RETURN_COL] = bench_ret
    factor_df[TARGET_COL] = factor_df[RAW_RETURN_COL] - factor_df[BENCHMARK_RETURN_COL]
    factor_df = factor_df.dropna(subset=[TARGET_COL]).copy()

    factor_df.to_csv(cache_path, index=False)
    print("saved", cache_path, factor_df.shape)
    gc.collect()
    return factor_df


def build_or_load_data():
    if (not REBUILD_DATA) and os.path.exists(DATA_PATH):
        return pd.read_csv(DATA_PATH)
    try:
        schedule = build_month_schedule(DATA_START_DATE, LABEL_END_DATE)
        print("schedule", schedule.shape)
        print(schedule.head())
        print(schedule.tail())
        valid_factors = precheck_valid_factors(schedule, ALL_CANDIDATE_FACTORS)
        parts = []
        for _, row in schedule.iterrows():
            part = build_one_month_panel(row, valid_factors)
            if part is not None and not part.empty:
                parts.append(part)
            gc.collect()
        if len(parts) == 0:
            raise ValueError("no monthly panels built")
        df = pd.concat(parts, ignore_index=True, sort=False)
        df.to_csv(DATA_PATH, index=False)
        return df
    except NameError as err:
        if os.path.exists(DATA_PATH):
            print("JoinQuant API unavailable; loading cached DATA_PATH instead.")
            return pd.read_csv(DATA_PATH)
        raise RuntimeError("JoinQuant API unavailable and DATA_PATH not found. Run this cell in JoinQuant or provide cache. err={}".format(err))


df_raw = build_or_load_data()
print("raw loaded", df_raw.shape)
print(df_raw[["rebalance_date", "feature_date", "next_date"]].agg(["min", "max"]))


In [ ]:
def normalize_df(df):
    out = df.copy()
    for c in ["rebalance_date", "feature_date", "next_date"]:
        if c in out.columns:
            out[c] = pd.to_datetime(out[c])
    for c in [TARGET_COL, RAW_RETURN_COL, BENCHMARK_RETURN_COL]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
    out = out.dropna(subset=["stock", "rebalance_date", "next_date", TARGET_COL]).copy()
    return out


def available_factor_cols(df):
    return [c for c in ALL_CANDIDATE_FACTORS if c in df.columns]


df_all = normalize_df(df_raw)
FACTOR_COLS_AVAILABLE = available_factor_cols(df_all)
FIXED_JQ_BASELINE_AVAILABLE = [c for c in FIXED_JQ_BASELINE_FACTORS if c in FACTOR_COLS_AVAILABLE]
print("df_all", df_all.shape)
print("months", df_all["rebalance_date"].nunique(), df_all["rebalance_date"].min(), df_all["rebalance_date"].max())
print("available candidate factors", len(FACTOR_COLS_AVAILABLE))
print(FACTOR_COLS_AVAILABLE)
print("fixed baseline available", len(FIXED_JQ_BASELINE_AVAILABLE))
print(FIXED_JQ_BASELINE_AVAILABLE)
print("missing fixed baseline factors", [c for c in FIXED_JQ_BASELINE_FACTORS if c not in FACTOR_COLS_AVAILABLE])


In [ ]:
def safe_rank_ic(x, y):
    tmp = pd.DataFrame({"x": x, "y": y}).replace([np.inf, -np.inf], np.nan).dropna()
    if len(tmp) < 30:
        return np.nan
    if tmp["x"].nunique() <= 1 or tmp["y"].nunique() <= 1:
        return np.nan
    return float(tmp["x"].rank(method="average").corr(tmp["y"].rank(method="average")))


def rank_pct_bottom(series):
    s = pd.to_numeric(series, errors="coerce")
    if s.notnull().sum() == 0:
        return pd.Series(0.0, index=s.index)
    fill_value = float(s.min()) - abs(float(s.min())) - 1.0
    return s.fillna(fill_value).rank(pct=True, method="average")


def factor_top_alpha_by_month(train_df, factor, direction, tail_months=None, topn=20):
    cols = ["rebalance_date", TARGET_COL, factor]
    tmp = train_df[cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if tmp.empty:
        return np.nan
    months = sorted(pd.to_datetime(tmp["rebalance_date"].unique()))
    if tail_months is not None and len(months) > tail_months:
        keep = set(months[-tail_months:])
        tmp = tmp[tmp["rebalance_date"].isin(keep)].copy()
    vals = []
    for dt, g in tmp.groupby("rebalance_date"):
        if len(g) < 30:
            continue
        g = g.copy()
        g["factor_score"] = g[factor].rank(pct=True, method="first") * float(direction)
        top = g.sort_values("factor_score", ascending=False).head(min(topn, len(g)))
        vals.append(float(top[TARGET_COL].mean()))
    return float(np.nanmean(vals)) if len(vals) else np.nan


def calc_factor_metrics(train_df, factor_cols):
    rows = []
    month_rows = []
    for factor in factor_cols:
        if factor not in train_df.columns:
            continue
        s = train_df[factor]
        coverage = float(s.notnull().mean())
        ic_rows = []
        if coverage >= MIN_FACTOR_COVERAGE:
            for dt, g in train_df[["rebalance_date", TARGET_COL, factor]].groupby("rebalance_date"):
                ic = safe_rank_ic(g[factor], g[TARGET_COL])
                if not pd.isnull(ic):
                    ic_rows.append({"rebalance_date": dt, "rank_ic": ic})
                    month_rows.append({"rebalance_date": dt, "factor": factor, "rank_ic": ic})
        if len(ic_rows) < MIN_MONTHS_FOR_FACTOR:
            rows.append({
                "factor": factor,
                "group": FACTOR_TO_GROUP.get(factor, "other"),
                "risk_bucket": risk_bucket(factor),
                "coverage": coverage,
                "months": len(ic_rows),
                "eligible_basic": False,
                "reject_base_reason": "low_coverage_or_months",
            })
            continue

        ic_df = pd.DataFrame(ic_rows).sort_values("rebalance_date")
        ic_mean_raw = float(ic_df["rank_ic"].mean())
        direction = 1.0 if ic_mean_raw >= 0 else -1.0
        ic_df["adj_ic"] = ic_df["rank_ic"] * direction
        adj_mean = float(ic_df["adj_ic"].mean())
        adj_std = float(ic_df["adj_ic"].std())
        ic_ir = adj_mean / adj_std if adj_std > 0 else 0.0
        pos_month_ratio = float((ic_df["adj_ic"] > 0).mean())
        recent_24_ic_adj = float(ic_df.tail(24)["adj_ic"].mean())
        recent_12_ic_adj = float(ic_df.tail(12)["adj_ic"].mean())
        prior_12_ic_adj = float(ic_df.iloc[-24:-12]["adj_ic"].mean()) if len(ic_df) >= 24 else np.nan
        recent_trend_ic = recent_12_ic_adj - prior_12_ic_adj if not pd.isnull(prior_12_ic_adj) else 0.0

        yearly = ic_df.copy()
        yearly["year"] = pd.to_datetime(yearly["rebalance_date"]).dt.year
        yearly_adj = yearly.groupby("year")["adj_ic"].mean()
        direction_stability = float((yearly_adj > 0).mean()) if len(yearly_adj) else np.nan
        worst_year_ic_adj = float(yearly_adj.min()) if len(yearly_adj) else np.nan

        top20_alpha_long = factor_top_alpha_by_month(train_df, factor, direction, tail_months=None, topn=20)
        top20_alpha_recent24 = factor_top_alpha_by_month(train_df, factor, direction, tail_months=24, topn=20)
        top20_alpha_recent12 = factor_top_alpha_by_month(train_df, factor, direction, tail_months=12, topn=20)

        eligible_core = bool(
            coverage >= 0.80 and len(ic_df) >= 48 and direction_stability >= 0.60 and
            pos_month_ratio >= 0.52 and worst_year_ic_adj >= -0.025 and ic_ir >= 0.10
        )
        eligible_governed = bool(
            coverage >= MIN_FACTOR_COVERAGE and len(ic_df) >= MIN_MONTHS_FOR_FACTOR and
            direction_stability >= 0.50 and worst_year_ic_adj >= -0.040
        )
        eligible_adaptive = bool(
            eligible_governed and recent_24_ic_adj > 0.0 and
            (recent_12_ic_adj > 0.0 or recent_trend_ic > 0.0)
        )

        rows.append({
            "factor": factor,
            "group": FACTOR_TO_GROUP.get(factor, "other"),
            "risk_bucket": risk_bucket(factor),
            "coverage": coverage,
            "months": len(ic_df),
            "direction": direction,
            "ic_mean_raw": ic_mean_raw,
            "long_ic_adj": adj_mean,
            "long_abs_ic": abs(ic_mean_raw),
            "long_ic_ir": ic_ir,
            "pos_month_ratio": pos_month_ratio,
            "direction_stability": direction_stability,
            "worst_year_ic_adj": worst_year_ic_adj,
            "recent_24_ic_adj": recent_24_ic_adj,
            "recent_12_ic_adj": recent_12_ic_adj,
            "recent_trend_ic": recent_trend_ic,
            "top20_alpha_long": top20_alpha_long,
            "top20_alpha_recent24": top20_alpha_recent24,
            "top20_alpha_recent12": top20_alpha_recent12,
            "eligible_basic": True,
            "eligible_core": eligible_core,
            "eligible_governed": eligible_governed,
            "eligible_adaptive": eligible_adaptive,
            "reject_base_reason": "",
        })
    metric_df = pd.DataFrame(rows)
    monthly_ic_df = pd.DataFrame(month_rows)
    if metric_df.empty:
        return metric_df, monthly_ic_df
    for flag_col in ["eligible_basic", "eligible_core", "eligible_governed", "eligible_adaptive"]:
        if flag_col not in metric_df.columns:
            metric_df[flag_col] = False
        metric_df[flag_col] = metric_df[flag_col].fillna(False).astype(bool)

    for col in [
        "long_ic_adj", "long_ic_ir", "pos_month_ratio", "direction_stability", "worst_year_ic_adj",
        "top20_alpha_long", "recent_24_ic_adj", "top20_alpha_recent24", "recent_12_ic_adj",
        "top20_alpha_recent12", "recent_trend_ic",
    ]:
        if col not in metric_df.columns:
            metric_df[col] = np.nan
        metric_df[col + "_rank"] = rank_pct_bottom(metric_df[col])

    metric_df["long_quality_score"] = (
        0.25 * metric_df["long_ic_adj_rank"] +
        0.20 * metric_df["long_ic_ir_rank"] +
        0.15 * metric_df["pos_month_ratio_rank"] +
        0.15 * metric_df["direction_stability_rank"] +
        0.15 * metric_df["worst_year_ic_adj_rank"] +
        0.10 * metric_df["top20_alpha_long_rank"]
    )
    metric_df["medium_quality_score"] = (
        0.60 * metric_df["recent_24_ic_adj_rank"] +
        0.40 * metric_df["top20_alpha_recent24_rank"]
    )
    metric_df["recent_quality_score"] = (
        0.45 * metric_df["recent_12_ic_adj_rank"] +
        0.35 * metric_df["top20_alpha_recent12_rank"] +
        0.20 * metric_df["recent_trend_ic_rank"]
    )
    metric_df["governed_score"] = (
        LONG_SCORE_WEIGHT * metric_df["long_quality_score"] +
        MEDIUM_SCORE_WEIGHT * metric_df["medium_quality_score"] +
        RECENT_SCORE_WEIGHT * metric_df["recent_quality_score"]
    )
    metric_df["core_score"] = 0.80 * metric_df["long_quality_score"] + 0.20 * metric_df["medium_quality_score"]
    metric_df["adaptive_score"] = (
        0.45 * metric_df["long_quality_score"] +
        0.35 * metric_df["medium_quality_score"] +
        0.20 * metric_df["recent_quality_score"]
    )
    for score_col in ["governed_score", "core_score", "adaptive_score"]:
        metric_df.loc[metric_df["eligible_basic"] != True, score_col] = -999.0
    return metric_df.sort_values("governed_score", ascending=False), monthly_ic_df


In [ ]:
def build_corr_matrix(train_df, factor_cols):
    cols = [c for c in unique_keep_order(factor_cols) if c in train_df.columns]
    if len(cols) == 0:
        return pd.DataFrame()
    ranked = train_df[cols].replace([np.inf, -np.inf], np.nan).rank(pct=True)
    return ranked.corr()


def max_corr_to_selected(candidate, selected, corr_mat):
    if len(selected) == 0 or corr_mat is None or corr_mat.empty:
        return 0.0, ""
    if candidate not in corr_mat.index:
        return 0.0, ""
    best_val = 0.0
    best_factor = ""
    for s in selected:
        if s in corr_mat.columns:
            val = corr_mat.loc[candidate, s]
            if not pd.isnull(val) and abs(float(val)) > abs(best_val):
                best_val = float(val)
                best_factor = s
    return abs(best_val), best_factor


def can_add_factor(row, selected, group_counts, defensive_count, corr_mat, group_max_quota, max_defensive_count):
    factor = row["factor"]
    group = row.get("group", "other")
    risk = row.get("risk_bucket", risk_bucket(factor))
    if factor in selected:
        return False, "already_selected"
    if group_counts.get(group, 0) >= group_max_quota.get(group, 99):
        return False, "group_quota_full"
    if risk == "defensive_low_vol_liq" and defensive_count >= max_defensive_count:
        return False, "defensive_low_vol_liq_quota_full"
    max_corr, corr_factor = max_corr_to_selected(factor, selected, corr_mat)
    if max_corr > MAX_ABS_CORR:
        return False, "high_corr_to_{}:{:.3f}".format(corr_factor, max_corr)
    return True, "selected"


def add_factor(row, selected, group_counts, defensive_count):
    factor = row["factor"]
    group = row.get("group", "other")
    risk = row.get("risk_bucket", risk_bucket(factor))
    selected.append(factor)
    group_counts[group] = group_counts.get(group, 0) + 1
    if risk == "defensive_low_vol_liq":
        defensive_count += 1
    return defensive_count


def select_with_governance(metric_df, train_df, score_col, target_count, min_count, eligibility_col, selector_name):
    if metric_df is None or metric_df.empty or score_col not in metric_df.columns:
        return [], pd.DataFrame()
    cand = metric_df[metric_df[eligibility_col] == True].copy()
    cand = cand.sort_values(score_col, ascending=False)
    if cand.empty:
        return [], pd.DataFrame()
    corr_mat = build_corr_matrix(train_df, list(cand["factor"]))
    selected = []
    group_counts = {}
    defensive_count = 0
    selection_rows = []

    # First satisfy minimum group quotas where possible.
    for group, quota in GROUP_MIN_QUOTA.items():
        group_cand = cand[cand["group"] == group].sort_values(score_col, ascending=False)
        for _, row in group_cand.iterrows():
            if group_counts.get(group, 0) >= quota:
                break
            ok, reason = can_add_factor(row, selected, group_counts, defensive_count, corr_mat, GROUP_MAX_QUOTA, MAX_DEFENSIVE_LOW_VOL_LIQ_COUNT)
            if ok:
                defensive_count = add_factor(row, selected, group_counts, defensive_count)
                selection_rows.append({
                    "selector_name": selector_name,
                    "factor": row["factor"],
                    "selected": True,
                    "phase": "min_group_quota",
                    "reason": "selected",
                    "score": row[score_col],
                    "group": row.get("group", "other"),
                    "risk_bucket": row.get("risk_bucket", risk_bucket(row["factor"])),
                })
            if len(selected) >= target_count:
                break
        if len(selected) >= target_count:
            break

    # Then fill by score with group/correlation/defensive constraints.
    for _, row in cand.iterrows():
        if len(selected) >= target_count:
            break
        ok, reason = can_add_factor(row, selected, group_counts, defensive_count, corr_mat, GROUP_MAX_QUOTA, MAX_DEFENSIVE_LOW_VOL_LIQ_COUNT)
        if ok:
            defensive_count = add_factor(row, selected, group_counts, defensive_count)
            selection_rows.append({
                "selector_name": selector_name,
                "factor": row["factor"],
                "selected": True,
                "phase": "score_fill",
                "reason": "selected",
                "score": row[score_col],
                "group": row.get("group", "other"),
                "risk_bucket": row.get("risk_bucket", risk_bucket(row["factor"])),
            })

    # If constraints make the pool too small, allow a controlled fallback by relaxing correlation only.
    if len(selected) < min_count:
        for _, row in cand.iterrows():
            if len(selected) >= min_count:
                break
            factor = row["factor"]
            group = row.get("group", "other")
            risk = row.get("risk_bucket", risk_bucket(factor))
            if factor in selected:
                continue
            if group_counts.get(group, 0) >= GROUP_MAX_QUOTA.get(group, 99):
                continue
            if risk == "defensive_low_vol_liq" and defensive_count >= MAX_DEFENSIVE_LOW_VOL_LIQ_COUNT:
                continue
            defensive_count = add_factor(row, selected, group_counts, defensive_count)
            selection_rows.append({
                "selector_name": selector_name,
                "factor": factor,
                "selected": True,
                "phase": "fallback_relax_corr",
                "reason": "selected_relaxed_corr",
                "score": row[score_col],
                "group": group,
                "risk_bucket": risk,
            })

    selected_set = set(selected)
    reject_rows = []
    selected_group_counts = dict(group_counts)
    selected_defensive = defensive_count
    for _, row in metric_df.iterrows():
        factor = row.get("factor", "")
        group = row.get("group", "other")
        risk = row.get("risk_bucket", risk_bucket(factor))
        if factor in selected_set:
            continue
        reason = "lower_score_or_not_needed"
        if row.get(eligibility_col, False) != True:
            reason = "not_{}".format(eligibility_col)
        elif selected_group_counts.get(group, 0) >= GROUP_MAX_QUOTA.get(group, 99):
            reason = "group_quota_full"
        elif risk == "defensive_low_vol_liq" and selected_defensive >= MAX_DEFENSIVE_LOW_VOL_LIQ_COUNT:
            reason = "defensive_low_vol_liq_quota_full"
        else:
            max_corr, corr_factor = max_corr_to_selected(factor, selected, corr_mat)
            if max_corr > MAX_ABS_CORR:
                reason = "high_corr_to_{}:{:.3f}".format(corr_factor, max_corr)
        reject_rows.append({
            "selector_name": selector_name,
            "factor": factor,
            "selected": False,
            "phase": "reject_review",
            "reason": reason,
            "score": row.get(score_col, np.nan),
            "group": group,
            "risk_bucket": risk,
        })
    rows = selection_rows + reject_rows
    return selected, pd.DataFrame(rows)


def select_governed_dynamic_pool(train_df, metric_df):
    return select_with_governance(
        metric_df=metric_df,
        train_df=train_df,
        score_col="governed_score",
        target_count=TARGET_FACTOR_COUNT,
        min_count=MIN_FACTOR_COUNT,
        eligibility_col="eligible_governed",
        selector_name="governed_dynamic_pool",
    )


def select_additional_with_existing(metric_df, train_df, existing, score_col, target_add_count, eligibility_col, selector_name):
    if metric_df is None or metric_df.empty or score_col not in metric_df.columns:
        return [], pd.DataFrame()
    existing = unique_keep_order(existing)
    cand = metric_df[(metric_df[eligibility_col] == True) & (~metric_df["factor"].isin(set(existing)))].copy()
    cand = cand.sort_values(score_col, ascending=False)
    if cand.empty or target_add_count <= 0:
        return [], pd.DataFrame()
    corr_mat = build_corr_matrix(train_df, list(cand["factor"]) + existing)
    selected_all = list(existing)
    added = []
    group_counts = {}
    defensive_count = 0
    for f in existing:
        group = FACTOR_TO_GROUP.get(f, "other")
        group_counts[group] = group_counts.get(group, 0) + 1
        if risk_bucket(f) == "defensive_low_vol_liq":
            defensive_count += 1
    rows = []
    for _, row in cand.iterrows():
        if len(added) >= target_add_count:
            break
        ok, reason = can_add_factor(row, selected_all, group_counts, defensive_count, corr_mat, GROUP_MAX_QUOTA, MAX_DEFENSIVE_LOW_VOL_LIQ_COUNT)
        if ok:
            defensive_count = add_factor(row, selected_all, group_counts, defensive_count)
            added.append(row["factor"])
            rows.append({
                "selector_name": selector_name,
                "factor": row["factor"],
                "selected": True,
                "phase": "add_to_existing",
                "reason": "selected",
                "score": row[score_col],
                "group": row.get("group", "other"),
                "risk_bucket": row.get("risk_bucket", risk_bucket(row["factor"])),
            })
    added_set = set(added)
    for _, row in cand.iterrows():
        factor = row["factor"]
        if factor in added_set:
            continue
        group = row.get("group", "other")
        risk = row.get("risk_bucket", risk_bucket(factor))
        reason = "lower_score_or_not_needed"
        if group_counts.get(group, 0) >= GROUP_MAX_QUOTA.get(group, 99):
            reason = "group_quota_full_after_core"
        elif risk == "defensive_low_vol_liq" and defensive_count >= MAX_DEFENSIVE_LOW_VOL_LIQ_COUNT:
            reason = "defensive_low_vol_liq_quota_full_after_core"
        else:
            max_corr, corr_factor = max_corr_to_selected(factor, selected_all, corr_mat)
            if max_corr > MAX_ABS_CORR:
                reason = "high_corr_to_{}:{:.3f}".format(corr_factor, max_corr)
        rows.append({
            "selector_name": selector_name,
            "factor": factor,
            "selected": False,
            "phase": "reject_review",
            "reason": reason,
            "score": row.get(score_col, np.nan),
            "group": group,
            "risk_bucket": risk,
        })
    return added, pd.DataFrame(rows)


def select_core_plus_adaptive_pool(train_df, metric_df):
    core, core_log = select_with_governance(
        metric_df=metric_df,
        train_df=train_df,
        score_col="core_score",
        target_count=CORE_TARGET_COUNT,
        min_count=min(12, CORE_TARGET_COUNT),
        eligibility_col="eligible_core",
        selector_name="core_stable_pool",
    )
    adaptive, adaptive_log = select_additional_with_existing(
        metric_df=metric_df,
        train_df=train_df,
        existing=core,
        score_col="adaptive_score",
        target_add_count=ADAPTIVE_TARGET_COUNT,
        eligibility_col="eligible_adaptive",
        selector_name="adaptive_recent_pool",
    )
    combined = unique_keep_order(core + adaptive)
    logs = [core_log, adaptive_log]
    # Conservative fill if too few adaptive factors survive; still respects existing exposure.
    if len(combined) < MIN_FACTOR_COUNT:
        fill, fill_log = select_additional_with_existing(
            metric_df=metric_df,
            train_df=train_df,
            existing=combined,
            score_col="governed_score",
            target_add_count=MIN_FACTOR_COUNT - len(combined),
            eligibility_col="eligible_governed",
            selector_name="core_plus_fallback_fill",
        )
        combined = unique_keep_order(combined + fill)
        logs.append(fill_log)
    log_df = pd.concat([x for x in logs if x is not None and not x.empty], ignore_index=True, sort=False) if len(logs) else pd.DataFrame()
    return combined[:TARGET_FACTOR_COUNT], log_df


def build_selected_factor_exposure(selector_name, window_row, factors):
    rows = []
    for f in factors:
        rows.append({
            "selector_name": selector_name,
            "window_name": window_row["window_name"],
            "test_year": int(window_row["test_year"]),
            "factor": f,
            "group": FACTOR_TO_GROUP.get(f, "other"),
            "risk_bucket": risk_bucket(f),
        })
    return pd.DataFrame(rows)


In [ ]:
def prepare_xy(df, feature_cols, target_col):
    feature_cols = [c for c in unique_keep_order(feature_cols) if c in df.columns]
    data = df.replace([np.inf, -np.inf], np.nan).copy()
    X = data[feature_cols].copy()
    fill_values = X.median().to_dict()
    X = X.fillna(pd.Series(fill_values)).fillna(0.0)
    y = pd.to_numeric(data[target_col], errors="coerce")
    valid = y.notnull()
    return X.loc[valid], y.loc[valid], fill_values


def train_lgb_model(train_df, feature_cols, target_col):
    if lgb is None:
        raise RuntimeError("lightgbm is not available")
    X, y, fill_values = prepare_xy(train_df, feature_cols, target_col)
    if len(X) == 0:
        raise ValueError("empty training rows")
    dtrain = lgb.Dataset(X, label=y, feature_name=list(feature_cols))
    model = lgb.train(dict(LGB_PARAMS), dtrain, num_boost_round=NUM_BOOST_ROUND)
    imp = pd.DataFrame({
        "feature": list(feature_cols),
        "gain_importance": model.feature_importance(importance_type="gain"),
        "split_importance": model.feature_importance(importance_type="split"),
    })
    return model, fill_values, imp, len(X)


def score_with_model(model, fill_values, df, feature_cols):
    X = df[feature_cols].replace([np.inf, -np.inf], np.nan).copy()
    X = X.fillna(pd.Series(fill_values)).fillna(0.0)
    return np.asarray(model.predict(X[feature_cols])).reshape(-1)


def max_drawdown(ret_series):
    if len(ret_series) == 0:
        return np.nan
    nav = (1.0 + pd.Series(ret_series).fillna(0.0)).cumprod()
    peak = nav.cummax()
    dd = nav / peak - 1.0
    return float(dd.min())


def calc_cum_excess(monthly):
    if monthly is None or monthly.empty:
        return np.nan
    cum_ret = float((1.0 + monthly["raw_return_1m"].fillna(0.0)).prod() - 1.0)
    cum_bench = float((1.0 + monthly["benchmark_csi800_1m"].fillna(0.0)).prod() - 1.0)
    return float((1.0 + cum_ret) / (1.0 + cum_bench) - 1.0) if (1.0 + cum_bench) != 0 else np.nan


def drop_top_month_excess(monthly, n):
    if monthly is None or monthly.empty or len(monthly) <= n:
        return np.nan
    keep = monthly.sort_values("alpha_1m", ascending=False).iloc[n:].copy()
    return calc_cum_excess(keep)


def evaluate_topn(score_df, strategy_name, topn):
    rows = []
    for dt, g in score_df.groupby("rebalance_date"):
        g = g.sort_values("score", ascending=False)
        top = g.head(min(topn, len(g)))
        raw_ret = float(top[RAW_RETURN_COL].mean()) if len(top) else np.nan
        bench_ret = float(top[BENCHMARK_RETURN_COL].iloc[0]) if len(top) else np.nan
        alpha = raw_ret - bench_ret
        rows.append({
            "strategy_name": strategy_name,
            "portfolio_profile": "top{}".format(topn),
            "rebalance_date": dt,
            "raw_return_1m": raw_ret,
            "benchmark_csi800_1m": bench_ret,
            "alpha_1m": alpha,
            "target_count": int(len(top)),
            "targets": ",".join(list(top["stock"])),
        })
    monthly = pd.DataFrame(rows).sort_values("rebalance_date")
    if monthly.empty:
        return monthly, {}
    cum_ret = float((1.0 + monthly["raw_return_1m"].fillna(0.0)).prod() - 1.0)
    cum_bench = float((1.0 + monthly["benchmark_csi800_1m"].fillna(0.0)).prod() - 1.0)
    cum_excess = calc_cum_excess(monthly)
    summary = {
        "strategy_name": strategy_name,
        "portfolio_profile": "top{}".format(topn),
        "months": int(len(monthly)),
        "cum_ret": cum_ret,
        "cum_csi800": cum_bench,
        "cum_excess_csi800": cum_excess,
        "mean_monthly_excess": float(monthly["alpha_1m"].mean()),
        "win_rate": float((monthly["alpha_1m"] > 0).mean()),
        "max_drawdown": max_drawdown(monthly["raw_return_1m"]),
        "drop_top1_excess": drop_top_month_excess(monthly, 1),
        "drop_top3_excess": drop_top_month_excess(monthly, 3),
    }
    return monthly, summary


def calc_oos_rank_ic(score_df):
    rows = []
    if score_df is None or score_df.empty:
        return pd.DataFrame()
    for (strategy_name, window_name, dt), g in score_df.groupby(["strategy_name", "window_name", "rebalance_date"]):
        ic = safe_rank_ic(g["score"], g[TARGET_COL])
        rows.append({
            "strategy_name": strategy_name,
            "window_name": window_name,
            "rebalance_date": dt,
            "rank_ic": ic,
        })
    return pd.DataFrame(rows)


def build_windows(df):
    rows = []
    for test_year in EVAL_YEARS:
        train_start_year = test_year - TRAIN_START_YEAR_OFFSET
        train_start = pd.Timestamp("{}-01-01".format(train_start_year))
        train_end = pd.Timestamp("{}-12-31".format(test_year - 1))
        test_start = pd.Timestamp("{}-01-01".format(test_year))
        test_end = pd.Timestamp("{}-12-31".format(test_year))
        train_df = df[(df["rebalance_date"] >= train_start) & (df["rebalance_date"] <= train_end)].copy()
        test_df = df[(df["rebalance_date"] >= test_start) & (df["rebalance_date"] <= test_end)].copy()
        if train_df.empty or test_df.empty:
            continue
        rows.append({
            "test_year": int(test_year),
            "window_name": "train{}_{}_test{}".format(train_start_year, test_year - 1, test_year),
            "train_start": train_start,
            "train_end": train_end,
            "test_start": test_start,
            "test_end": test_end,
        })
    return pd.DataFrame(rows)


windows_df = build_windows(df_all)
print(windows_df)


In [ ]:
def run_one_model(train_df, test_df, feature_cols, strategy_name, window_row):
    feature_cols = [c for c in unique_keep_order(feature_cols) if c in train_df.columns and c in test_df.columns]
    if len(feature_cols) == 0:
        return pd.DataFrame(), pd.DataFrame(), {}
    model, fill_values, importance, train_rows = train_lgb_model(train_df, feature_cols, TARGET_COL)
    base_cols = ["stock", "rebalance_date", "feature_date", "next_date", RAW_RETURN_COL, BENCHMARK_RETURN_COL, TARGET_COL]
    score_df = test_df[base_cols].copy()
    score_df["score"] = score_with_model(model, fill_values, test_df, feature_cols)
    score_df["strategy_name"] = strategy_name
    score_df["window_name"] = window_row["window_name"]
    score_df["test_year"] = int(window_row["test_year"])
    score_df["feature_count"] = len(feature_cols)

    importance["strategy_name"] = strategy_name
    importance["window_name"] = window_row["window_name"]
    importance["test_year"] = int(window_row["test_year"])

    meta = {
        "strategy_name": strategy_name,
        "window_name": window_row["window_name"],
        "test_year": int(window_row["test_year"]),
        "train_start": window_row["train_start"],
        "train_end": window_row["train_end"],
        "test_start": window_row["test_start"],
        "test_end": window_row["test_end"],
        "train_rows": int(train_rows),
        "test_rows": int(len(test_df)),
        "feature_count": int(len(feature_cols)),
        "features": ",".join(feature_cols),
        "score_type": "lgb_regression_alpha_1m",
    }

    if EXPORT_MODELS:
        bundle = {
            "objective": "v54_governed_dynamic_factor_pool_lgb",
            "strategy_name": strategy_name,
            "window_name": window_row["window_name"],
            "target_col": TARGET_COL,
            "base_model": model,
            "base_feature_cols": list(feature_cols),
            "base_fill_values": dict(fill_values),
            "base_params": dict(LGB_PARAMS),
            "fixed_iter": int(NUM_BOOST_ROUND),
            "stock_num": 10,
            "benchmark": BENCHMARK_INDEX,
            "requires_v4_feature_adapter": False,
        }
        pkl_path = os.path.join(MODEL_EXPORT_DIR, "model_{}_{}.pkl".format(strategy_name, window_row["window_name"]))
        with open(pkl_path, "wb") as f:
            pickle.dump(bundle, f, protocol=2)
        meta["model_path"] = pkl_path
    return score_df, importance, meta


score_parts = []
importance_parts = []
meta_rows = []
factor_metric_parts = []
monthly_ic_parts = []
selection_log_parts = []
selected_exposure_parts = []
monthly_parts = []
summary_rows = []

for _, win in windows_df.iterrows():
    print("running", win["window_name"])
    train_df = df_all[(df_all["rebalance_date"] >= win["train_start"]) & (df_all["rebalance_date"] <= win["train_end"])].copy()
    test_df = df_all[(df_all["rebalance_date"] >= win["test_start"]) & (df_all["rebalance_date"] <= win["test_end"])].copy()

    metric_df, monthly_ic_df = calc_factor_metrics(train_df, FACTOR_COLS_AVAILABLE)
    if not metric_df.empty:
        metric_tag = metric_df.copy()
        metric_tag["window_name"] = win["window_name"]
        metric_tag["test_year"] = int(win["test_year"])
        factor_metric_parts.append(metric_tag)
    if not monthly_ic_df.empty:
        monthly_tag = monthly_ic_df.copy()
        monthly_tag["window_name"] = win["window_name"]
        monthly_tag["test_year"] = int(win["test_year"])
        monthly_ic_parts.append(monthly_tag)

    governed_features, governed_log = select_governed_dynamic_pool(train_df, metric_df)
    core_adaptive_features, core_adaptive_log = select_core_plus_adaptive_pool(train_df, metric_df)
    fixed_features = list(FIXED_JQ_BASELINE_AVAILABLE)

    print("fixed", len(fixed_features), fixed_features)
    print("governed", len(governed_features), governed_features)
    print("core_plus_adaptive", len(core_adaptive_features), core_adaptive_features)

    for selector_name, features, log_df in [
        ("fixed_jq_baseline", fixed_features, pd.DataFrame()),
        ("governed_dynamic_pool", governed_features, governed_log),
        ("core_plus_adaptive_pool", core_adaptive_features, core_adaptive_log),
    ]:
        exp = build_selected_factor_exposure(selector_name, win, features)
        if not exp.empty:
            selected_exposure_parts.append(exp)
        if log_df is not None and not log_df.empty:
            tag = log_df.copy()
            tag["window_name"] = win["window_name"]
            tag["test_year"] = int(win["test_year"])
            selection_log_parts.append(tag)

    model_specs = [
        ("fixed_jq_baseline_lgb", fixed_features),
        ("governed_dynamic_pool_lgb", governed_features),
        ("core_plus_adaptive_lgb", core_adaptive_features),
    ]

    for strategy_name, features in model_specs:
        score_df_part, imp_df, meta = run_one_model(train_df, test_df, features, strategy_name, win)
        if score_df_part.empty:
            continue
        score_parts.append(score_df_part)
        importance_parts.append(imp_df)
        meta_rows.append(meta)
        for topn in TOP_LIST:
            monthly_eval, summary = evaluate_topn(score_df_part, strategy_name, topn)
            if not monthly_eval.empty:
                monthly_eval["window_name"] = win["window_name"]
                monthly_eval["test_year"] = int(win["test_year"])
                monthly_parts.append(monthly_eval)
                summary["window_name"] = win["window_name"]
                summary["test_year"] = int(win["test_year"])
                summary_rows.append(summary)
    gc.collect()

score_df = pd.concat(score_parts, ignore_index=True, sort=False) if score_parts else pd.DataFrame()
importance_df = pd.concat(importance_parts, ignore_index=True, sort=False) if importance_parts else pd.DataFrame()
model_meta_df = pd.DataFrame(meta_rows)
factor_metrics_df = pd.concat(factor_metric_parts, ignore_index=True, sort=False) if factor_metric_parts else pd.DataFrame()
monthly_factor_ic_df = pd.concat(monthly_ic_parts, ignore_index=True, sort=False) if monthly_ic_parts else pd.DataFrame()
selection_log_df = pd.concat(selection_log_parts, ignore_index=True, sort=False) if selection_log_parts else pd.DataFrame()
selected_exposure_df = pd.concat(selected_exposure_parts, ignore_index=True, sort=False) if selected_exposure_parts else pd.DataFrame()
monthly_df = pd.concat(monthly_parts, ignore_index=True, sort=False) if monthly_parts else pd.DataFrame()
summary_df = pd.DataFrame(summary_rows).sort_values(["test_year", "portfolio_profile", "strategy_name"]) if summary_rows else pd.DataFrame()
rank_ic_df = calc_oos_rank_ic(score_df)

print("summary")
print(summary_df)


In [ ]:
def build_baseline_delta(summary):
    if summary is None or summary.empty:
        return pd.DataFrame()
    base = summary[summary["strategy_name"] == "fixed_jq_baseline_lgb"].copy()
    cols = ["window_name", "test_year", "portfolio_profile", "cum_excess_csi800", "mean_monthly_excess", "max_drawdown", "drop_top1_excess", "drop_top3_excess"]
    base = base[cols].copy()
    base.columns = ["window_name", "test_year", "portfolio_profile", "baseline_cum_excess", "baseline_mean_monthly_excess", "baseline_max_drawdown", "baseline_drop_top1_excess", "baseline_drop_top3_excess"]
    out = summary.merge(base, on=["window_name", "test_year", "portfolio_profile"], how="left")
    out["delta_cum_excess_vs_baseline"] = out["cum_excess_csi800"] - out["baseline_cum_excess"]
    out["delta_mean_monthly_excess_vs_baseline"] = out["mean_monthly_excess"] - out["baseline_mean_monthly_excess"]
    out["delta_max_drawdown_vs_baseline"] = out["max_drawdown"] - out["baseline_max_drawdown"]
    out["delta_drop_top1_vs_baseline"] = out["drop_top1_excess"] - out["baseline_drop_top1_excess"]
    out["delta_drop_top3_vs_baseline"] = out["drop_top3_excess"] - out["baseline_drop_top3_excess"]
    return out


def build_selection_group_summary(selected_exposure):
    if selected_exposure is None or selected_exposure.empty:
        return pd.DataFrame()
    rows = []
    keys = ["window_name", "test_year", "selector_name"]
    for key, g in selected_exposure.groupby(keys):
        group_counts = g.groupby("group").size().to_dict()
        risk_counts = g.groupby("risk_bucket").size().to_dict()
        row = {
            "window_name": key[0],
            "test_year": int(key[1]),
            "selector_name": key[2],
            "feature_count": int(len(g)),
            "defensive_low_vol_liq_count": int(risk_counts.get("defensive_low_vol_liq", 0)),
            "defensive_low_vol_liq_ratio": float(risk_counts.get("defensive_low_vol_liq", 0)) / float(max(1, len(g))),
        }
        for group_name in sorted(GROUP_MAX_QUOTA.keys()):
            row["group_count_{}".format(group_name)] = int(group_counts.get(group_name, 0))
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["test_year", "selector_name"])


def build_rank_ic_summary(rank_ic):
    if rank_ic is None or rank_ic.empty:
        return pd.DataFrame()
    rows = []
    tmp = rank_ic.copy()
    tmp["test_year"] = pd.to_datetime(tmp["rebalance_date"]).dt.year
    for key, g in tmp.groupby(["strategy_name", "window_name", "test_year"]):
        vals = pd.to_numeric(g["rank_ic"], errors="coerce").dropna()
        if len(vals) == 0:
            continue
        std = float(vals.std())
        rows.append({
            "strategy_name": key[0],
            "window_name": key[1],
            "test_year": int(key[2]),
            "months": int(len(vals)),
            "rank_ic_mean": float(vals.mean()),
            "rank_ic_ir": float(vals.mean()) / std if std > 0 else np.nan,
            "rank_ic_pos_ratio": float((vals > 0).mean()),
        })
    return pd.DataFrame(rows).sort_values(["test_year", "strategy_name"])


summary_delta_df = build_baseline_delta(summary_df)
selection_group_summary_df = build_selection_group_summary(selected_exposure_df)
rank_ic_summary_df = build_rank_ic_summary(rank_ic_df)

score_df.to_csv(os.path.join(OUT_DIR, "v54_scores.csv"), index=False)
importance_df.to_csv(os.path.join(OUT_DIR, "v54_importance.csv"), index=False)
model_meta_df.to_csv(os.path.join(OUT_DIR, "v54_model_meta.csv"), index=False)
factor_metrics_df.to_csv(os.path.join(OUT_DIR, "v54_factor_metrics_by_window.csv"), index=False)
monthly_factor_ic_df.to_csv(os.path.join(OUT_DIR, "v54_monthly_factor_ic.csv"), index=False)
selection_log_df.to_csv(os.path.join(OUT_DIR, "v54_selection_log.csv"), index=False)
selected_exposure_df.to_csv(os.path.join(OUT_DIR, "v54_selected_factor_exposure.csv"), index=False)
selection_group_summary_df.to_csv(os.path.join(OUT_DIR, "v54_selection_group_summary.csv"), index=False)
monthly_df.to_csv(os.path.join(OUT_DIR, "v54_monthly.csv"), index=False)
summary_df.to_csv(os.path.join(OUT_DIR, "v54_summary.csv"), index=False)
summary_delta_df.to_csv(os.path.join(OUT_DIR, "v54_summary_delta_vs_baseline.csv"), index=False)
rank_ic_df.to_csv(os.path.join(OUT_DIR, "v54_rank_ic.csv"), index=False)
rank_ic_summary_df.to_csv(os.path.join(OUT_DIR, "v54_rank_ic_summary.csv"), index=False)

print("saved outputs to", OUT_DIR)
print("summary delta vs baseline")
print(summary_delta_df.sort_values(["test_year", "portfolio_profile", "delta_cum_excess_vs_baseline"], ascending=[True, True, False]))
print("selection group summary")
print(selection_group_summary_df)
print("rank ic summary")
print(rank_ic_summary_df)


In [ ]:
# Export fixed baseline model bank for JoinQuant backtest.
# Run this cell after data/normalization/helper cells are available. It does not rerun V54 strategy comparison.
FIXED_BASELINE_EXPORT_DIR = os.path.join(OUT_DIR, "fixed_baseline_exported_models")
os.makedirs(FIXED_BASELINE_EXPORT_DIR, exist_ok=True)


def export_fixed_baseline_model_bank(df):
    export_rows = []
    feature_cols = [c for c in FIXED_JQ_BASELINE_FACTORS if c in df.columns]
    if len(feature_cols) == 0:
        raise ValueError("fixed baseline feature cols empty")
    for _, win in windows_df.iterrows():
        train_df = df[(df["rebalance_date"] >= win["train_start"]) & (df["rebalance_date"] <= win["train_end"])].copy()
        if train_df.empty:
            print("skip empty train window", win["window_name"])
            continue
        model, fill_values, importance, train_rows = train_lgb_model(train_df, feature_cols, TARGET_COL)
        train_start = pd.Timestamp(win["train_start"]).strftime("%Y%m%d")
        train_end = pd.Timestamp(win["train_end"]).strftime("%Y%m%d")
        test_year = int(win["test_year"])
        file_name = "model_csi800_lgb_v54_fixed_jq_baseline_train{}_{}_for{}.pkl".format(
            train_start, train_end, test_year
        )
        model_path = os.path.join(FIXED_BASELINE_EXPORT_DIR, file_name)
        bundle = {
            # Keep this objective for compatibility with the older direct-overlay loader.
            "objective": "v210_refit_fixed_iter_overlay",
            "research_version": "v54_fixed_jq_baseline_lgb_export",
            "strategy_name": "fixed_jq_baseline_lgb",
            "window_name": win["window_name"],
            "model_date_tag": "train{}_{}_for{}".format(train_start, train_end, test_year),
            "target_col": TARGET_COL,
            "base_model": model,
            "base_feature_cols": list(feature_cols),
            "base_fill_values": dict(fill_values),
            "base_params": dict(LGB_PARAMS),
            "fixed_iter": int(NUM_BOOST_ROUND),
            "overlay_mode": "direct",
            "overlay_weight": 0.0,
            "residual_model": None,
            "residual_feature_cols": [],
            "residual_fill_values": {},
            "top_n_candidates": 30,
            "stock_num": 10,
            "benchmark": BENCHMARK_INDEX,
            "universe_index": UNIVERSE_INDEX,
            "industry_cap_ratio": 0.20,
            "requires_v4_feature_adapter": False,
            "feature_source": "jqfactor_only",
            "train_start": pd.Timestamp(win["train_start"]).strftime("%Y-%m-%d"),
            "train_end": pd.Timestamp(win["train_end"]).strftime("%Y-%m-%d"),
            "test_year": test_year,
            "train_rows": int(train_rows),
        }
        with open(model_path, "wb") as f:
            pickle.dump(bundle, f, protocol=2)
        export_rows.append({
            "model_path": model_path,
            "model_file": file_name,
            "strategy_name": "fixed_jq_baseline_lgb",
            "window_name": win["window_name"],
            "test_year": test_year,
            "train_start": bundle["train_start"],
            "train_end": bundle["train_end"],
            "train_rows": int(train_rows),
            "feature_count": len(feature_cols),
            "features": ",".join(feature_cols),
        })
        importance_out = importance.copy()
        importance_out["model_file"] = file_name
        importance_out["window_name"] = win["window_name"]
        importance_out["test_year"] = test_year
        importance_out.to_csv(os.path.join(FIXED_BASELINE_EXPORT_DIR, file_name.replace(".pkl", "_importance.csv")), index=False)
        print("exported", model_path, "train_rows", train_rows, "features", len(feature_cols))
        gc.collect()
    export_df = pd.DataFrame(export_rows)
    export_df.to_csv(os.path.join(FIXED_BASELINE_EXPORT_DIR, "fixed_jq_baseline_model_manifest.csv"), index=False)
    return export_df


fixed_baseline_export_df = export_fixed_baseline_model_bank(df_all)
print("exported fixed baseline models:")
print(fixed_baseline_export_df)


## 结论填写区

运行后重点看：

1. `v54_summary_delta_vs_baseline.csv`：`governed_dynamic_pool_lgb` 和 `core_plus_adaptive_lgb` 是否在 Top10/Top20 的多数年份超过 `fixed_jq_baseline_lgb`。
2. `v54_selection_group_summary.csv`：低波/低流动性防御因子占比是否被控制住。
3. `v54_selection_log.csv`：被剔除的因子是否主要因为相关性、group quota、防御暴露过重，而不是低级缺列问题。
4. `v54_rank_ic_summary.csv`：新模型是否真的提高 OOS RankIC，而不是只靠少数月份收益。
5. `drop_top1_excess/drop_top3_excess`：去掉头部月份后是否还保留有效超额。

判定标准：
- 如果新模型只在一个年份或一个 profile 胜出，视为 candidate，不作为主策略。
- 如果 Top10、Top20、RankIC、drop-top stress 方向一致优于 baseline，才认为动态治理因子池有效。
- 如果仍打不过 baseline，后续停止动态选因子主线，回到固定因子池精修和组合层优化。